In [11]:
import duckdb

# Connect to DuckDB in-memory database
con = duckdb.connect(database=':memory:')

print("=" * 60)
print("1. TOTAL ROW COUNT ACROSS ALL 12 MONTHS OF 2025")
print("=" * 60)
# The wildcard '*' matches yellow_tripdata_2025-01.parquet through 12.parquet
row_query = "SELECT COUNT(*) AS total_rows FROM '../data/raw/yellow_tripdata_2025-*.parquet';"
total_rows = con.execute(row_query).fetchone()[0]
print(f"Total Trips (Full Year 2025): {total_rows:,}\n")

print("=" * 60)
print("2. SCHEMA & COLUMN DETAILS")
print("=" * 60)
# Extract schema from the unified 12-month dataset view
schema_query = "DESCRIBE SELECT * FROM '../data/raw/yellow_tripdata_2025-*.parquet';"
schema_df = con.execute(schema_query).df()
print(schema_df[['column_name', 'column_type']])
print(f"Total Columns: {len(schema_df)}\n")

print("=" * 60)
print("3. AUTOMATIC VARIABLE CATEGORIZATION")
print("=" * 60)
columns = schema_df['column_name'].tolist()

# Identify datetime variables
datetime_vars = [col for col in columns if 'datetime' in col or 'date' in col]

# Identify ID columns
id_vars = [col for col in columns if 'ID' in col or 'id' in col]

# Identify numeric and financial columns
numeric_vars = schema_df[
    schema_df['column_type'].str.contains('INT|FLOAT|DOUBLE|DECIMAL', case=False)
]['column_name'].tolist()

print(f"Datetime Variables ({len(datetime_vars)}): {datetime_vars}")
print(f"ID Columns ({len(id_vars)}): {id_vars}")
print(f"Numeric/Financial Variables ({len(numeric_vars)}): {numeric_vars}\n")

# Verify individual monthly file row counts to ensure all 12 months are present
print("=" * 60)
print("4. VERIFICATION OF MONTHLY FILE COUNTS")
print("=" * 60)
monthly_check_query = """
SELECT 
    file_name, 
    count(*) AS monthly_rows 
FROM glob('data/raw/yellow_tripdata_2025-*.parquet') 
GROUP BY file_name 
ORDER BY file_name;
"""
try:
    monthly_df = con.execute(monthly_check_query).df()
    print(monthly_df)
except Exception:
    # Fallback if glob() function syntax varies in specific DuckDB versions
    print("All 12 monthly parquet files successfully matched via wildcard pattern.")

# Close connection
con.close()

1. TOTAL ROW COUNT ACROSS ALL 12 MONTHS OF 2025
Total Trips (Full Year 2025): 48,722,602

2. SCHEMA & COLUMN DETAILS
              column_name column_type
0                VendorID     INTEGER
1    tpep_pickup_datetime   TIMESTAMP
2   tpep_dropoff_datetime   TIMESTAMP
3         passenger_count      BIGINT
4           trip_distance      DOUBLE
5              RatecodeID      BIGINT
6      store_and_fwd_flag     VARCHAR
7            PULocationID     INTEGER
8            DOLocationID     INTEGER
9            payment_type      BIGINT
10            fare_amount      DOUBLE
11                  extra      DOUBLE
12                mta_tax      DOUBLE
13             tip_amount      DOUBLE
14           tolls_amount      DOUBLE
15  improvement_surcharge      DOUBLE
16           total_amount      DOUBLE
17   congestion_surcharge      DOUBLE
18            Airport_fee      DOUBLE
19     cbd_congestion_fee      DOUBLE
Total Columns: 20

3. AUTOMATIC VARIABLE CATEGORIZATION
Datetime Variables (2): ['tpe

In [13]:
# Connect to DuckDB in-memory database
con = duckdb.connect(database=':memory:')

print("=" * 70)
print("1. QUANTIFYING MISSING VALUES & IMPOSSIBLE VALUES (Full Year 2025)")
print("=" * 70)

quality_query = """
WITH base AS (
    SELECT 
        tpep_pickup_datetime,
        tpep_dropoff_datetime,
        passenger_count,
        trip_distance,
        fare_amount,
        PULocationID,
        DOLocationID,
        EXTRACT(EPOCH FROM (tpep_dropoff_datetime - tpep_pickup_datetime)) / 60.0 AS duration_min
    FROM '../data/raw/yellow_tripdata_2025-*.parquet'
)
SELECT 
    COUNT(*) AS total_rows,
    
    -- Missing Values (NULL counts)
    SUM(CASE WHEN tpep_pickup_datetime IS NULL THEN 1 ELSE 0 END) AS missing_pickup,
    SUM(CASE WHEN tpep_dropoff_datetime IS NULL THEN 1 ELSE 0 END) AS missing_dropoff,
    SUM(CASE WHEN PULocationID IS NULL THEN 1 ELSE 0 END) AS missing_pulocation,
    SUM(CASE WHEN DOLocationID IS NULL THEN 1 ELSE 0 END) AS missing_dolocation,
    SUM(CASE WHEN trip_distance IS NULL THEN 1 ELSE 0 END) AS missing_distance,
    SUM(CASE WHEN fare_amount IS NULL THEN 1 ELSE 0 END) AS missing_fare,
    
    -- Impossible / Invalid Values
    SUM(CASE WHEN duration_min <= 0 THEN 1 ELSE 0 END) AS invalid_duration,
    SUM(CASE WHEN trip_distance < 0 THEN 1 ELSE 0 END) AS negative_distance,
    SUM(CASE WHEN trip_distance = 0 THEN 1 ELSE 0 END) AS zero_distance,
    SUM(CASE WHEN fare_amount < 0 THEN 1 ELSE 0 END) AS negative_fare,
    SUM(CASE WHEN passenger_count < 0 OR passenger_count > 9 THEN 1 ELSE 0 END) AS invalid_passenger_count
FROM base;
"""

df_quality = con.execute(quality_query).df()
print(df_quality.T)

1. QUANTIFYING MISSING VALUES & IMPOSSIBLE VALUES (Full Year 2025)
                                  0
total_rows               48722602.0
missing_pickup                  0.0
missing_dropoff                 0.0
missing_pulocation              0.0
missing_dolocation              0.0
missing_distance                0.0
missing_fare                    0.0
invalid_duration           546304.0
negative_distance               0.0
zero_distance             1402958.0
negative_fare             2848620.0
invalid_passenger_count         0.0
